# MiMo-V2.6 Heretic 9B (uncensored) on Colab → OpenAI-compatible endpoint

- Model: `maternion/mimo-v2.6-heretic:9b` (Ollama) — MiMo-V2.6-Distill-Qwen-9B, abliterated
- Context: **77K** (78,848 tokens) · KV cache: **q8_0** · Flash Attention: on
- **Thinking: optional, default OFF** (set in Config; override per request)
- Public URL via Cloudflare quick tunnel + API-key auth proxy
- Use from anywhere: `base_url = https://xxxx.trycloudflare.com/v1`

**Runtime → Change runtime type → T4 GPU**, then Run all.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

## 1. Install Ollama, cloudflared, proxy deps

In [ ]:
%%bash
set -e
apt-get -qq update && apt-get -qq install -y zstd pciutils lshw > /dev/null
curl -fsSL https://ollama.com/install.sh | sh > /tmp/ollama_install.log 2>&1 || (tail -20 /tmp/ollama_install.log; exit 1)
ollama --version
wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
chmod +x /usr/local/bin/cloudflared
cloudflared --version
pip -q install fastapi uvicorn httpx openai

## 2. Config
- **THINKING**: server-wide default. `off` = model answers directly (default). `on` = model reasons first.
  - Per request you can override it: send `"think": true/false` or `"reasoning_effort": "none" | "low" | "medium" | "high"`.
- **QUANT**: `Q4_K_M` (5.6 GB, safest on a T4), `q6_K` (7.4 GB), `q8_0` (9.5 GB, tight with 77K ctx on 16 GB).

In [ ]:
import os, secrets

THINKING = "off"      #@param ["off", "on"]
QUANT    = "Q4_K_M"   #@param ["Q4_K_M", "q6_K", "q8_0"]

NUM_CTX  = 77 * 1024  # 78848 tokens (~77K)
KV_CACHE = "q8_0"     # f16 | q8_0 | q4_0

# Tags from https://ollama.com/maternion/mimo-v2.6-heretic
#  off -> "-instruct" tags (temp 0.6, top_k 20, presence 1.5)
#  on  -> "-thinking" tags (temp 1.0, top_k 0)
variant   = "thinking" if THINKING == "on" else "instruct"
suffix    = "" if QUANT == "Q4_K_M" else f"-{QUANT}"
MODEL_SRC   = f"maternion/mimo-v2.6-heretic:9b-{variant}{suffix}"
MODEL_ALIAS = "mimo-v2.6-heretic"   # name you'll use in the OpenAI API

# Leave "" to auto-generate a random key. Put your own string to keep a fixed key.
API_KEY = ""
API_KEY = API_KEY or "sk-" + secrets.token_urlsafe(24)

print("Source  :", MODEL_SRC)
print("Alias   :", MODEL_ALIAS)
print("Thinking:", THINKING, "(default)")
print("Context :", NUM_CTX, "| KV cache:", KV_CACHE)

## 3. Start Ollama server (flash attention + q8 KV cache + 77K ctx)

In [ ]:
import subprocess, time, requests

env = os.environ.copy()
env.update({
    "OLLAMA_HOST": "127.0.0.1:11434",
    "OLLAMA_FLASH_ATTENTION": "1",        # required for KV cache quantization
    "OLLAMA_KV_CACHE_TYPE": KV_CACHE,
    "OLLAMA_CONTEXT_LENGTH": str(NUM_CTX),
    "OLLAMA_NUM_PARALLEL": "1",           # parallel requests multiply context memory
    "OLLAMA_MAX_LOADED_MODELS": "1",
    "OLLAMA_KEEP_ALIVE": "-1",            # keep model loaded
})

subprocess.run(["pkill", "-f", "ollama serve"], stderr=subprocess.DEVNULL)
time.sleep(1)
ollama_log = open("/tmp/ollama.log", "w")
ollama_proc = subprocess.Popen(["ollama", "serve"], env=env, stdout=ollama_log, stderr=subprocess.STDOUT)

for _ in range(60):
    try:
        print("Ollama up, version:", requests.get("http://127.0.0.1:11434/api/version").json()["version"])
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama failed to start:\n" + open("/tmp/ollama.log").read()[-3000:])

## 4. Pull model and create alias with `num_ctx` baked in

In [ ]:
!ollama pull {MODEL_SRC}

modelfile = f"""FROM {MODEL_SRC}
PARAMETER num_ctx {NUM_CTX}
"""
open("/tmp/Modelfile", "w").write(modelfile)
!ollama create {MODEL_ALIAS} -f /tmp/Modelfile
!ollama list

## 5. Warm up the model and verify GPU + context

In [ ]:
r = requests.post("http://127.0.0.1:11434/api/generate",
                  json={"model": MODEL_ALIAS, "prompt": "hi", "stream": False, "think": False,
                        "options": {"num_predict": 8}}, timeout=900)
print(r.status_code)
!ollama ps
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv
!grep -iE "KvSize|kv_cache|flash|offload" /tmp/ollama.log | tail -8

## 6. API-key auth proxy + thinking switch
Ollama has no auth, and its OpenAI endpoint only turns thinking off via `reasoning_effort: "none"`. The proxy handles both:
- no `think` / `reasoning_effort` in the request → uses the **THINKING** default from Config
- `"think": false` → `reasoning_effort: "none"` (off) · `"think": true` → `"medium"` · `"think": "high"` → `"high"`
- an explicit `reasoning_effort` or `reasoning` field is passed through untouched

In [ ]:
%%writefile /tmp/proxy.py
import os, json, httpx
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from starlette.background import BackgroundTask

KEY = os.environ.get("API_KEY", "")
DEFAULT_THINK = os.environ.get("DEFAULT_THINK", "off")   # "off" | "on"
app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
client = httpx.AsyncClient(base_url="http://127.0.0.1:11434", timeout=httpx.Timeout(None))

def apply_thinking(body: bytes) -> bytes:
    try:
        data = json.loads(body)
    except Exception:
        return body
    if not isinstance(data, dict):
        return body
    think = data.pop("think", None)
    if think is not None:                                  # per-request override
        t = str(think).lower()
        if t in ("false", "off", "none", "0"):
            data["reasoning_effort"] = "none"
        elif t in ("true", "on", "1"):
            data["reasoning_effort"] = "medium"
        else:
            data["reasoning_effort"] = t                   # low / medium / high
    elif "reasoning_effort" not in data and "reasoning" not in data:
        if DEFAULT_THINK == "off":                         # server default
            data["reasoning_effort"] = "none"
    return json.dumps(data).encode()

@app.api_route("/{path:path}", methods=["GET", "POST", "DELETE"])
async def proxy(path: str, request: Request):
    if KEY and request.headers.get("authorization", "") != f"Bearer {KEY}":
        return JSONResponse({"error": {"message": "Invalid API key", "type": "invalid_request_error"}}, status_code=401)
    body = await request.body()
    if request.method == "POST" and path.rstrip("/") == "v1/chat/completions":
        body = apply_thinking(body)
    # Only forward necessary headers to avoid host-mismatch and connection errors
    headers = {}
    for k, v in request.headers.items():
        if k.lower() in ("content-type", "accept", "accept-encoding"):
            headers[k] = v

    req = client.build_request(request.method, "/" + path, content=body, headers=headers, params=request.query_params)
    try:
        r = await client.send(req, stream=True)
        out_headers = {"content-type": r.headers.get("content-type", "application/json")}
        if "content-encoding" in r.headers:
            out_headers["content-encoding"] = r.headers["content-encoding"]
        return StreamingResponse(
            r.aiter_raw(), status_code=r.status_code, headers=out_headers,
            background=BackgroundTask(r.aclose),
        )
    except Exception as e:
        return JSONResponse({"error": {"message": str(e), "type": "proxy_error"}}, status_code=500)

In [ ]:
subprocess.run(["pkill", "-f", "uvicorn proxy:app"], stderr=subprocess.DEVNULL)
penv = os.environ.copy(); penv["API_KEY"] = API_KEY; penv["DEFAULT_THINK"] = THINKING
proxy_log = open("/tmp/proxy.log", "w")
proxy_proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "proxy:app", "--host", "127.0.0.1", "--port", "8000", "--log-level", "warning"],
    cwd="/tmp", env=penv, stdout=proxy_log, stderr=subprocess.STDOUT)
time.sleep(3)
print("proxy status:", requests.get("http://127.0.0.1:8000/v1/models",
      headers={"Authorization": f"Bearer {API_KEY}"}).status_code)

## 7. Public tunnel (Cloudflare quick tunnel, no account needed)

In [ ]:
import re

subprocess.run(["pkill", "-f", "cloudflared"], stderr=subprocess.DEVNULL)
cf_log = open("/tmp/cloudflared.log", "w")
cf_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
                           stdout=cf_log, stderr=subprocess.STDOUT)
PUBLIC_URL = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/tmp/cloudflared.log").read())
    if m:
        PUBLIC_URL = m.group(0); break
    time.sleep(1)
assert PUBLIC_URL, open("/tmp/cloudflared.log").read()[-2000:]

print("=" * 60)
print("Base URL :", PUBLIC_URL + "/v1")
print("API key  :", API_KEY)
print("Model    :", MODEL_ALIAS)
print("Thinking :", THINKING, "(default; override per request with \"think\": true/false)")
print("=" * 60)
print(f"""
curl {PUBLIC_URL}/v1/chat/completions \\
  -H "Authorization: Bearer {API_KEY}" -H "Content-Type: application/json" \\
  -d '{{"model":"{MODEL_ALIAS}","messages":[{{"role":"user","content":"Hello!"}}]}}'

# force thinking on/off for one request:
#   add  "think": true   or   "think": false   to the JSON body
""")

## 8. Test through the public URL (default mode, then the opposite via per-request override)

In [ ]:
time.sleep(5)  # let the tunnel DNS propagate
from openai import OpenAI
oa = OpenAI(base_url=PUBLIC_URL + "/v1", api_key=API_KEY)

def ask(prompt, **extra):
    stream = oa.chat.completions.create(
        model=MODEL_ALIAS, stream=True, extra_body=extra or None,
        messages=[{"role": "user", "content": prompt}],
    )
    thinking_started = False
    for chunk in stream:
        if not chunk.choices:
            continue
        d = chunk.choices[0].delta
        reasoning = getattr(d, "reasoning", None) or getattr(d, "reasoning_content", None)
        if reasoning:
            if not thinking_started:
                print("[thinking] ", end=""); thinking_started = True
            print(reasoning, end="", flush=True)
        if d.content:
            if thinking_started:
                print("\n[answer] ", end=""); thinking_started = False
            print(d.content, end="", flush=True)
    print("\n")

print(f"--- default (thinking {THINKING}) ---")
ask("What is 17 * 23? Answer briefly.")

print(f"--- override: think={THINKING == 'off'} ---")
ask("What is 17 * 23? Answer briefly.", think=(THINKING == "off"))

## 9. Keep alive / logs
Run this last cell to keep Colab busy and the tunnel up. Stop it to free the runtime.
Free Colab disconnects after idle/12h; the tunnel URL changes every time you re-run the notebook.
To change the thinking default, change `THINKING` in Config and Run all again.

In [ ]:
import datetime
try:
    while True:
        ok = requests.get("http://127.0.0.1:11434/api/version", timeout=5).ok
        print(datetime.datetime.now().strftime("%H:%M:%S"), "ollama:", ok, "|", PUBLIC_URL + "/v1", end="\r")
        time.sleep(60)
except KeyboardInterrupt:
    print("\nStopped monitor (server still running).")